It's better to test on the converge of the model.

In [167]:
import tensorflow as tf
import keras
from keras.utils import to_categorical
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [168]:
from keras import Sequential
from keras.layers import Dense, BatchNormalization, Dropout
from keras.losses import CategoricalCrossentropy
from keras.optimizers import Adam

In [169]:
from deel.influenciae.common import InfluenceModel, ExactIHVP
from deel.influenciae.influence import FirstOrderInfluenceCalculator
from deel.influenciae.utils import ORDER
from deel.influenciae.trac_in import TracIn

In [170]:
import random
from keras.optimizers import SGD

In [171]:
from sklearn.datasets import make_classification

Train_Size: 1000, 2000, 4000, 8000, 16000  
Feature_Size: 10, 20, 40, 80, 160

In [172]:
# train_pool = 16000
test_size = 500
# n_features=10
seed=42
ratios = [(9,1), (8,2), (7,3), (6,4), (5,5)]

In [173]:
ds, ds_info = tfds.load('diamonds', split='train', with_info=True, as_supervised=True)
df = tfds.as_dataframe(ds, ds_info)
print(df)

       features/carat  features/clarity  features/color  features/cut  \
0                1.26                 2               4             2   
1                0.80                 3               4             4   
2                0.56                 4               2             4   
3                1.51                 3               6             1   
4                0.33                 6               5             4   
...               ...               ...             ...           ...   
53935            1.02                 2               4             2   
53936            0.93                 2               4             3   
53937            0.30                 4               5             4   
53938            0.36                 3               2             4   
53939            0.70                 1               2             2   

       features/depth  features/table  features/x  features/y  features/z  \
0           60.599998            60.0        6

In [174]:
median_price = df["price"].median()
df["label"] = (df["price"] > median_price).astype(int)
df = df.drop(columns=['price'])

In [175]:
df['id'] = np.arange(1, len(df) + 1)
print(df)

       features/carat  features/clarity  features/color  features/cut  \
0                1.26                 2               4             2   
1                0.80                 3               4             4   
2                0.56                 4               2             4   
3                1.51                 3               6             1   
4                0.33                 6               5             4   
...               ...               ...             ...           ...   
53935            1.02                 2               4             2   
53936            0.93                 2               4             3   
53937            0.30                 4               5             4   
53938            0.36                 3               2             4   
53939            0.70                 1               2             2   

       features/depth  features/table  features/x  features/y  features/z  \
0           60.599998            60.0        6

In [176]:
df['label'].value_counts()

label
0    26985
1    26955
Name: count, dtype: int64

In [177]:
exact_size = 8000

In [178]:
cur_ratio = ratios[4]
print(cur_ratio)

(5, 5)


In [179]:
major, minor = cur_ratio

In [180]:
df0 = df[df.label == 0]  
df1 = df[df.label == 1] 

In [181]:
t0 = int(exact_size * major / (major + minor))
t1 = exact_size - t0 
print(t0,t1)

4000 4000


In [182]:
s0 = df0.sample(n=t0, random_state=seed)
s1 = df1.sample(n=t1, random_state=seed)

In [183]:
df = pd.concat([s0, s1], axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)

In [184]:
print(df)
print(df["label"].value_counts())

      features/carat  features/clarity  features/color  features/cut  \
0               0.33                 6               1             4   
1               0.26                 5               3             4   
2               0.30                 3               0             3   
3               0.41                 4               3             3   
4               1.07                 3               1             4   
...              ...               ...             ...           ...   
7995            1.16                 1               3             3   
7996            1.71                 1               2             4   
7997            0.38                 2               3             4   
7998            0.71                 5               4             2   
7999            0.90                 2               5             3   

      features/depth  features/table  features/x  features/y  features/z  \
0          60.900002            56.0        4.46        4.4

In [185]:
id_label_df = df[["id", "label"]].copy()
print(id_label_df)
id_label_df.to_csv("ClassImbalance_Diamonds/5_5_class_labelIDs.csv",index = False)

         id  label
0     11439      0
1     24565      0
2     27356      0
3     29515      0
4     28058      1
...     ...    ...
7995  51575      1
7996  48670      1
7997  38341      0
7998  42760      1
7999  36494      1

[8000 rows x 2 columns]


In [186]:
n0 = test_size //2
n1 = test_size - n0

In [187]:
g = df.groupby("label", group_keys=False)
test_df = pd.concat([
    g.get_group(0).sample(n=n0, random_state=42, replace=False),
    g.get_group(1).sample(n=n1, random_state=42, replace=False),
]).sample(frac=1, random_state=42)

train_df = df.drop(test_df.index).reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

In [188]:
print(test_df.groupby("label").get_group(0))
print(test_df["label"].value_counts())

     features/carat  features/clarity  features/color  features/cut  \
1              0.52                 3               0             3   
3              0.39                 2               1             3   
4              0.34                 3               1             4   
7              0.32                 6               3             4   
8              0.38                 2               1             4   
..              ...               ...             ...           ...   
492            0.30                 5               1             3   
493            0.32                 7               5             3   
494            0.32                 2               0             2   
495            0.60                 1               0             3   
499            0.34                 7               3             3   

     features/depth  features/table  features/x  features/y  features/z  \
1         62.400002            58.0        5.09        5.13        3.19 

In [189]:
X_train = train_df.drop(columns=["label"])
y_train = train_df["label"]
IDs = X_train["id"].values.reshape(-1, 1).astype(np.float32)
IDs = IDs  / 1e10

X_train = X_train.drop(columns=["id"]).values.astype(np.float32)
X_train = np.hstack((X_train, IDs))
y_train = to_categorical(y_train.values,num_classes=2)

print(X_train)

[[3.3000e-01 6.0000e+00 1.0000e+00 ... 4.4800e+00 2.7200e+00 1.1439e-06]
 [2.6000e-01 5.0000e+00 3.0000e+00 ... 4.1500e+00 2.5600e+00 2.4565e-06]
 [3.0000e-01 3.0000e+00 0.0000e+00 ... 4.3200e+00 2.6200e+00 2.7356e-06]
 ...
 [3.8000e-01 2.0000e+00 3.0000e+00 ... 4.6800e+00 2.8600e+00 3.8341e-06]
 [7.1000e-01 5.0000e+00 4.0000e+00 ... 5.7200e+00 3.5800e+00 4.2760e-06]
 [9.0000e-01 2.0000e+00 5.0000e+00 ... 6.1900e+00 3.8300e+00 3.6494e-06]]


In [190]:
X_test = test_df.drop(columns=["label"])
y_test = test_df["label"]
IDs = X_test["id"].values.reshape(-1, 1).astype(np.float32)
IDs = IDs  / 1e10

X_test = X_test.drop(columns=["id"]).values.astype(np.float32)
X_test = np.hstack((X_test, IDs))
y_test = to_categorical(y_test.values,num_classes=2)

print(X_test)

[[1.2100e+00 2.0000e+00 2.0000e+00 ... 6.8500e+00 4.2500e+00 7.7840e-07]
 [5.2000e-01 3.0000e+00 0.0000e+00 ... 5.1300e+00 3.1900e+00 2.0663e-06]
 [1.5900e+00 4.0000e+00 6.0000e+00 ... 7.5400e+00 4.5600e+00 4.8120e-07]
 ...
 [1.6000e+00 1.0000e+00 1.0000e+00 ... 7.5500e+00 4.7000e+00 2.9842e-06]
 [9.0000e-01 3.0000e+00 2.0000e+00 ... 6.2200e+00 3.7100e+00 4.3238e-06]
 [3.4000e-01 7.0000e+00 3.0000e+00 ... 4.4700e+00 2.7200e+00 1.2445e-06]]


In [191]:
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))

# Fold 1

In [192]:
from tensorflow.keras.regularizers import l2

In [193]:
seed_value = 42
random.seed(seed_value)
np.random.seed(seed_value)
tf.random.set_seed(seed_value)

model = Sequential([
    Dense(16, activation='relu', input_shape=(X_train.shape[1],)),  
    BatchNormalization(momentum=0.9),
    Dropout(0.0),
    Dense(8, activation='relu'),
    Dense(y_train.shape[1])
])
loss_fn = CategoricalCrossentropy(from_logits=True)
optimizer = SGD(learning_rate=0.001, momentum=0.9)
model.compile(loss=loss_fn, optimizer=optimizer, metrics=['accuracy'])

epochs = 500
unreduced_loss_fn = CategoricalCrossentropy(from_logits=True, reduction=tf.keras.losses.Reduction.NONE)
model_list = []
model_list.append(InfluenceModel(model, start_layer=-1, loss_function=unreduced_loss_fn))
for i in range(epochs):
  model.fit(train_ds.batch(256), epochs=1, validation_data=test_ds.batch(256), verbose=2)
  model_list.append(InfluenceModel(model, start_layer=-1, loss_function=unreduced_loss_fn))
base_loss, acc = model.evaluate(test_ds.batch(32), verbose=2)
print(base_loss)

30/30 - 0s - loss: 0.6388 - accuracy: 0.6341 - val_loss: 0.5812 - val_accuracy: 0.7060 - 497ms/epoch - 17ms/step
30/30 - 0s - loss: 0.5604 - accuracy: 0.7489 - val_loss: 0.5432 - val_accuracy: 0.8120 - 205ms/epoch - 7ms/step
30/30 - 0s - loss: 0.4968 - accuracy: 0.8280 - val_loss: 0.4939 - val_accuracy: 0.8460 - 192ms/epoch - 6ms/step
30/30 - 0s - loss: 0.4373 - accuracy: 0.8689 - val_loss: 0.4423 - val_accuracy: 0.8740 - 145ms/epoch - 5ms/step
30/30 - 0s - loss: 0.3837 - accuracy: 0.8899 - val_loss: 0.3858 - val_accuracy: 0.8720 - 147ms/epoch - 5ms/step
30/30 - 0s - loss: 0.3370 - accuracy: 0.9047 - val_loss: 0.3416 - val_accuracy: 0.8880 - 149ms/epoch - 5ms/step
30/30 - 0s - loss: 0.2977 - accuracy: 0.9155 - val_loss: 0.3073 - val_accuracy: 0.8880 - 157ms/epoch - 5ms/step
30/30 - 0s - loss: 0.2647 - accuracy: 0.9247 - val_loss: 0.2947 - val_accuracy: 0.8920 - 137ms/epoch - 5ms/step
30/30 - 0s - loss: 0.2375 - accuracy: 0.9296 - val_loss: 0.2888 - val_accuracy: 0.8840 - 142ms/epoch - 

IF

In [194]:
train_ids = []
test_ids = []
train_samples_np = np.array([x.numpy() for x, y in train_ds])
train_ids = [round(sample[-1] * 1e10) for sample in train_samples_np]

In [195]:
num_test_samples = len(test_ds)
num_train_samples = len(train_ids)
test_ids = []

influence_model = model_list[-1]
ihvp_calculator = ExactIHVP(influence_model, train_ds.batch(16))
influence_calculator = FirstOrderInfluenceCalculator(influence_model, train_ds, ihvp_calculator)

influence_matrix = np.zeros((num_test_samples, num_train_samples))

samples_to_explain = test_ds.take(num_test_samples).batch(1)
explanation_ds = influence_calculator.top_k(samples_to_explain, train_ds.batch(16), k=num_train_samples, order=ORDER.DESCENDING)
for test_idx,((sample, label), top_k_values, top_k_samples) in enumerate(explanation_ds.as_numpy_iterator()):
    test_sample_id = round(sample[0][-1] * 1e10)
    test_ids.append(test_sample_id)
    influential_ids = [round(s[-1] * 1e10) for s in top_k_samples[0]] 
    influence_scores = top_k_values[0]
    id_to_index = {train_id: idx for idx, train_id in enumerate(train_ids)}
    for inf_id, score in zip(influential_ids, influence_scores):
        if inf_id in id_to_index:
            influence_matrix[test_idx, id_to_index[inf_id]] = score

flattened_row = np.median(influence_matrix, axis=0).reshape(1, -1)
df = pd.DataFrame({'Train_ID': train_ids, 'Score': flattened_row.flatten()})
print(df)

      Train_ID         Score
0        11439  1.868813e-03
1        24565  2.853596e-04
2        27356  5.364553e-04
3        29515  1.218391e-03
4        28058  2.017613e-07
...        ...           ...
7495     51575  3.042660e-06
7496     48670  9.320552e-12
7497     38341  2.088249e-03
7498     42760  9.325851e-03
7499     36494  2.121805e-03

[7500 rows x 2 columns]


TC

In [196]:
num_test_samples = len(test_ds)
num_train_samples = len(train_ids)
test_ids = []

TracIn_matrix = np.zeros((num_test_samples, num_train_samples))
influence_calculator = TracIn(
    model_list, 0.001
)
samples_to_explain = test_ds.take(num_test_samples).batch(1)
explanation_ds = influence_calculator.top_k(samples_to_explain, train_ds.batch(16), k=num_train_samples, order=ORDER.DESCENDING)
for test_idx,((sample, label), top_k_values, top_k_samples) in enumerate(explanation_ds.as_numpy_iterator()):
    test_sample_id = round(sample[0][-1] * 1e10)
    test_ids.append(test_sample_id)
    influential_ids = [round(s[-1] * 1e10) for s in top_k_samples[0]] 
    influence_scores = top_k_values[0]
    id_to_index = {train_id: idx for idx, train_id in enumerate(train_ids)}
    for inf_id, score in zip(influential_ids, influence_scores):
        if inf_id in id_to_index:
            TracIn_matrix[test_idx, id_to_index[inf_id]] = score

flattened_row = np.median(TracIn_matrix, axis=0).reshape(1, -1)
TracIn_df = pd.DataFrame({'Train_ID': train_ids, 'Score': flattened_row.flatten()})
print(TracIn_df)

      Train_ID         Score
0        11439  1.455761e-05
1        24565  3.262551e-06
2        27356  7.681573e-06
3        29515  1.936030e-05
4        28058 -1.284862e-10
...        ...           ...
7495     51575 -2.034189e-09
7496     48670 -3.059713e-15
7497     38341  1.187625e-05
7498     42760 -6.126181e-05
7499     36494 -4.876160e-06

[7500 rows x 2 columns]


To have more direct view of that, try to match the ranking of two dataframe.

In [197]:
df_sorted = df.sort_values(by="Score", ascending=False).reset_index(drop=True)

TracIn_sorted = TracIn_df.sort_values(by="Score", ascending=False).reset_index(drop=True)

In [198]:
TracIn_sorted.to_csv("ClassImbalance_Diamonds/TC_5_5_class.csv",index = False)
df_sorted.to_csv("ClassImbalance_Diamonds/IF_5_5_class.csv",index = False)